In [ ]:
from fits_io import scan_dataset, update_headers, dataset_metadata, object_in_fov
from fits_io import identify_objects, cleanup_intermediate_files 
from fits_io import flag_object_in_fov, load_dataset_objects
from reduccion import run_reduction, plot_reduction
from astrometria import run_astrometry
from stars_catalog import generate_refcat, plot_catalog_on_image
from combine import process_and_combine_images, plot_combined, remove_aligment_tempfiles
import pandas as pd
from pathlib import Path
import os
from utils import load_config, Tee, load_target_coordinates
from datetime import datetime
import sys
from autophot.autophot_main import run_automatic_autophot
import tempfile
import shutil
from autophot.prep_input import load





# -----------------------------------------------------------------------------
# Load settings
# -----------------------------------------------------------------------------

config_file="config.yaml"
cfg = load_config(config_file)

BASE = Path(cfg["paths"]["BASE"])
data_path = Path(BASE, cfg["paths"]["data_dir"])
object_path = Path(data_path, cfg["paths"]["objects_dir"])
night = cfg["night"]
night_dir = Path(data_path, night)
objects_csv = Path(object_path, "objetos.csv")
images_file_name = cfg["paths"]["images_data_file"]
images_file = Path(night_dir, images_file_name)

gain = cfg["instrument"]["gain"]
rdnoise = cfg["instrument"]["rdnoise"]

steps = cfg["steps"]

print(f"Running pipeline in night dir: {night_dir}")
# -------------------------------------------------------------------------
# Redirect stdout/stderr to a single log file (append)
# -------------------------------------------------------------------------
log_file = night_dir / "pipeline.log"
#sys.stdout = Tee(log_file)
#sys.stderr = sys.stdout

print("\n"*2 + "="*80)
print(f"New pipeline run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)


# -------------------------------------------------------------------------
# 1) Scan dataset
# -------------------------------------------------------------------------
dataset = scan_dataset(night_dir)

# -------------------------------------------------------------------------
# 2) Update headers
# -------------------------------------------------------------------------
if steps.get("update_headers", False):
    print("→ Updating FITS headers")
    update_headers(dataset["all"], gain=gain, rdnoise=rdnoise)

# -------------------------------------------------------------------------
# 3) Export metadata
# -------------------------------------------------------------------------
if steps.get("export_metadata", False):
    print(f"→ Exporting metadata to {images_file}")
    metadata_file = dataset_metadata(dataset, night_dir, output_file=images_file,
                                         objects_csv=objects_csv)
    # Load objects list
    objects = load_dataset_objects(night_dir, images_file)
    
    # Create a folder in objects dir for each object
    for objname in objects:
        obj_dir = Path(object_path, objname)
        obj_dir.mkdir(parents=True, exist_ok=True)


# -------------------------------------------------------------------------
# 4) Reduction
# -------------------------------------------------------------------------
if steps.get("reduction", False):
    print("→ Running reduction")

    reduction_cfg = cfg.get("reduction", {})
    reduction_result = run_reduction(
        dataset=dataset,
        zero_correction=reduction_cfg.get("zero_correction", True),
        flat_correction=reduction_cfg.get("flat_correction", True),
        dark_correction=reduction_cfg.get("dark_correction", False),
    )
else:
    reduction_result = None
    
# -------------------------------------------------------------------------
# 4.a) Plot de reducción
# -------------------------------------------------------------------------
if cfg["qc"].get("reduction_images", False):
    print("→ [QC] Generating reduction plots")
    for objname in objects:
        print(f"   → Object: {objname}")

        plot_reduction(
            dataset=images_file,
            night_dir=night_dir,
            objname=objname,
            output_name=f"reduction_{objname}.png",
            show=False,
            overwrite=True
        )
    print("✓ [QC] Reduction plots generated")

# -------------------------------------------------------------------------
# 5) Astrometry
# -------------------------------------------------------------------------
if steps.get("astrometry", False):
    print("→ Running astrometry")

    astro_cfg = cfg.get("astrometry", {})
    run_astrometry(
        dataset=dataset,
        output_dir=night_dir,
        api_key=astro_cfg["api_key"],
        overwrite=astro_cfg.get("overwrite", True),
    )
    dataset = scan_dataset(night_dir)
    metadata_file = dataset_metadata(dataset, night_dir,
                                    output_file=images_file)
# -------------------------------------------------------------------------
# 5.5) Images contains its object?
# -------------------------------------------------------------------------
if steps.get("contains_obj", False):
    print("→ Flagging not contained images")
    df = flag_object_in_fov(metadata_file,
                       objects_csv,
                       night_dir,
                       overwrite=False)
    metadata_file = dataset_metadata(dataset, night_dir,
                                    output_file=images_file)

# -------------------------------------------------------------------------
# 6) Combination
# -------------------------------------------------------------------------
if steps.get("combine", False):
    print("→ Combining images per object and filter")
    
    filters = cfg.get("filters", ["I", "V"])
    for objname in objects:
        print(f"   → Object: {objname}")
        print(f"      → Generating reference catalog")
        # Load metadata for the object
        ra, dec = load_target_coordinates(objname, objects_csv)
        alig_cfg = cfg.get("aligment", {})["catalog"]
        # Load image path for the object
        ds = pd.read_csv(Path(night_dir, images_file))
        img_files = ds[(ds["OBJECT"]==objname)&(ds["ASTROMET"]=="yes")]["FILENAME"].values
        img_path = night_dir / img_files[0]

        alig_cat_path = generate_refcat(
            objname=objname, ra_center=ra, dec_center=dec,
            img_path=img_path, objects_dir=object_path,
            fov_frac=alig_cfg.get("fov_frac", 0.3),
            min_mag=alig_cfg.get("min_mag", 9),
            max_mag=alig_cfg.get("max_mag", 12),
            use_catalogs=alig_cfg.get("use_catalogs", "all"),
            plot=alig_cfg.get("plot", False),
            overwrite=alig_cfg.get("overwrite", False),
        )

        for filt in filters:
            print(f"      → Filter: {filt}")
            images_to_combine = [night_dir / im for im in img_files if im[-14].upper() == filt.upper()]
            if len(images_to_combine) == 0:
                print(f"         ! No images found for filter {filt}. Skipping.")
                continue
            print(f"         ✓ Found {len(images_to_combine)} images for filter {filt}.")
            combined_im = process_and_combine_images(images_to_combine, objname,  filt, object_path, night_dir, objects_csv)
            # Plot aligment reference catalog on combined images (unnecessary step)
            plot_catalog_on_image(fits_file=combined_im,
                                  catalog=alig_cat_path,
                                  obj_ra=ra,
                                  obj_dec=dec,
                                  out_png=Path(night_dir, f"{objname}{filt}_comb_alig_cat.png"),
                                  title=f"{objname} – Aligment catalog")
            if cfg["combine"].get("remove_temp_files", False):
                print("      → Removing temporary files")
                removed = remove_aligment_tempfiles(filt, night_dir)
    dataset = scan_dataset(night_dir)
    metadata_file = dataset_metadata(dataset, night_dir,
                                    output_file=images_file)

if cfg["qc"].get("combined_images", False):
    print("→ Generating combined plots")
    for objname in objects:
        print(f"   → Object: {objname}")
        image_files = [file for file in dataset["images_combined"] if objname in file.name]
        if len(image_files) == 0:
            print(f"      ! No combined images found for object {objname}. Skipping.")
            continue  
        plot_combined(objname, image_files,  night_dir, show=False)
        
        
if steps.get("photometry", False):
    print("→ Photometry on science images")
    phot_cfg = cfg["photometry"]
    img_files = []
    if phot_cfg["on_exp"]:
        img_files += dataset["images_astro"]
    if phot_cfg["on_comb"]:
        img_files += dataset["images_combined"]
         
    for objname in objects:
        print(f"   → Object: {objname}")
        print(f"      → Generating reference catalog")
        ra, dec = load_target_coordinates(objname, objects_csv)
        phot_cat_cfg = cfg["photometry"]["catalog"]
        img_path = night_dir / img_files[0]
        phot_cat_path = generate_refcat(
            objname=objname, ra_center=ra, dec_center=dec,
            img_path=img_path, objects_dir=object_path,
            fov_frac=phot_cat_cfg.get("fov_frac", 0.3),
            min_mag=phot_cat_cfg.get("min_mag", 9),
            max_mag=phot_cat_cfg.get("max_mag", 12),
            max_mag_err=phot_cat_cfg.get("max_mag_err", 0.25),
            use_catalogs=phot_cat_cfg.get("use_catalogs", False),
            plot=phot_cat_cfg.get("plot", False),
            type="phot",
            overwrite=phot_cat_cfg.get("overwrite", False)
        )
        print(f"      → Plotting catalog on combined image")
        combined_im = [f for f in dataset["images_combined"] if objname in f.name][0]
        plot_catalog_on_image(fits_file=combined_im,
                        catalog=phot_cat_path,
                        obj_ra=ra,
                        obj_dec=dec,
                        out_png=Path(object_path, objname, f"{objname}_phot_cat.png"),
                        title=f"{objname} – Aligment catalog")
        print("   → Run photometry")
        #autophot_input = cfg["photometry"]["autophot"]
        temp_dir = Path(night_dir, "phot")
        os.makedirs(temp_dir, exist_ok=True)
        refcat_csv = Path(object_path, objname, f"{objname}_phot_cat.csv")
        
        with tempfile.TemporaryDirectory(prefix=f"autophot_{objname}_", dir=night_dir) as temp_dir_str:
            temp_dir = Path(temp_dir_str)
            autophot_input = load()
            
            obj_images = [p for p in img_files if objname.lower() in p.stem.lower()]
            
            for p in obj_images:
                (temp_dir / p.name).symlink_to(p.resolve())  # o copy2
        
            wdir = Path(object_path, objname, "autophot").resolve()
            wdir.mkdir(parents=True, exist_ok=True)
            autophot_input["wdir"] = str(wdir)
            autophot_input["fits_dir"] = str(temp_dir)

            autophot_input["target_name"] = objname
            autophot_input["target_ra"] = ra/15
            autophot_input["target_dec"] = dec

            autophot_input["catalog"]["catalog_custom_fpath"] = str(refcat_csv) 

            try:
                run_automatic_autophot(autophot_input)
            except Exception as e:
                print(f"         Error AutoPhOT {objname}: {e}")

   
print("✓ Pipeline finished successfully")


Running pipeline in night dir: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/20250823bis


New pipeline run: 2026-01-22 14:19:42
→ Updating FITS headers
→ Exporting metadata to /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/20250823bis/images_data.csv


   Processing FITS files: 100%|██████████| 498/498 [00:00<00:00, 83240.88it/s]

→ Photometry on science images
   → Object: OGLE-2025-BLG-0451
      → Generating reference catalog
         Sobrescribiendo archivo /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0451/OGLE-2025-BLG-0451_phot_cat.csv
         FOV diagonal: 13.5' → Search radius: 5.4'
            • Catalog Gaia3 → 38 refs
      ✓ Archivo /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0451/OGLE-2025-BLG-0451_phot_cat.csv generado correctamente.
      → Plotting catalog on combined image


      ✓ Plot saved: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0451/OGLE-2025-BLG-0451_phot_cat.png
   → Run photometry
Default input loaded in from: 
/home/knowogrodski/Documentos/observation_HSH/process_HSH_SN/autophot/autophot/databases/default_input.yml

        _       _       ___ _  _    _____
       /_\ _  _| |_ ___| _ \ || |__|_   _|
      / _ \ || |  _/ _ \  _/ __ / _ \| |
     /_/ \_\_,_|\__\___/_| |_||_\___/|_|
    
     ---------------------------------------
        Automated Photometry of Transients
        S. J. Brennan et al. 2021 
        Please provide feedback/bugs to:
        Email: sean.brennan2@ucdconnect.ie
    ---------------------------------------
Directory of fits file: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/20250823bis/autophot_OGLE-2025-BLG-0451_xrsnwasy
Found Telescopes:

 - HSH

-> Telescope check complete

Checking Filter keywords and database

-> Filter check complete

Che

In [4]:
config_file="config.yaml"
from utils import load_config
cfg = load_config(config_file)
from pathlib import Path

BASE = Path(cfg["paths"]["BASE"])
data_path = Path(BASE, cfg["paths"]["data_dir"])
object_path = Path(data_path, cfg["paths"]["objects_dir"])
night = cfg["night"]
night_dir = Path(data_path, night)
objects_csv = Path(object_path, "objetos.csv")
images_file_name = cfg["paths"]["images_data_file"]
images_file = Path(night_dir, images_file_name)
wdir = os.path.join(BASE, "outputs", night, objname)
autophot_input["wdir"] = str(night_dir)
autophot_input["fits_dir"] = str(temp_dir)

gain = cfg["instrument"]["gain"]
rdnoise = cfg["instrument"]["rdnoise"]
print("DEBUG PATHS:")
print("  night_dir:", night_dir.resolve())
print("  object_path:", object_path.resolve())
print("  temp_dir:", temp_dir.resolve())
print("  refcat_csv:", refcat_csv.resolve())
print("  autophot_input['wdir']:", autophot_input['wdir'])
print("  autophot_input['fits_dir']:", autophot_input['fits_dir'])
print("  autophot_input['catalog']['catalog_custom_fpath']:", autophot_input['catalog']['catalog_custom_fpath'])
print("  telescope.yml path que AutoPhOT usa (del log):", "ver el log de AutoPhOT")
print("  os.getcwd() actual:", os.getcwd())

DEBUG PATHS:
  night_dir: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/20250823bis
  object_path: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos
  temp_dir: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/20250823bis/autophot_OGLE-2025-GD-0001_i3m5y4pw
  refcat_csv: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-GD-0001/OGLE-2025-GD-0001_phot_cat.csv
  autophot_input['wdir']: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/20250823bis
  autophot_input['fits_dir']: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/20250823bis/autophot_OGLE-2025-GD-0001_i3m5y4pw
  autophot_input['catalog']['catalog_custom_fpath']: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-GD-0001/OGLE-2025-GD-0001_phot_cat.csv
  telescope.yml path que AutoPhOT usa (del log): ver el log de AutoPhOT
  os.getcwd()

In [9]:
refcat_csv

PosixPath('/home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0451/OGLE-2025-BLG-0451_phot_cat.csv')

In [ ]:
def comparar_dicts(d1, d2, path=""):
    diferencias = []
    
    # Claves solo en d1
    for k in d1:
        if k not in d2:
            diferencias.append(f"{path}{k} → solo en dict1")
    
    # Claves solo en d2
    for k in d2:
        if k not in d1:
            diferencias.append(f"{path}{k} → solo en dict2")
    
    # Claves en ambos → comparar valores
    for k in set(d1) & set(d2):
        v1, v2 = d1[k], d2[k]
        current_path = f"{path}{k}."
        
        if type(v1) != type(v2):
            diferencias.append(f"{path}{k} → tipos distintos: {type(v1)} vs {type(v2)}")
        
        elif isinstance(v1, dict):
            diferencias.extend(comparar_dicts(v1, v2, current_path))
        
        elif isinstance(v1, list):
            if len(v1) != len(v2):
                diferencias.append(f"{path}{k} → longitudes distintas: {len(v1)} vs {len(v2)}")
            else:
                for i, (x, y) in enumerate(zip(v1, v2)):
                    if x != y:
                        diferencias.append(f"{current_path}[{i}] → {x} ≠ {y}")
        
        elif v1 != v2:
            diferencias.append(f"{path}{k} → {v1} ≠ {v2}")
    
    return diferencias


# Uso
difs = comparar_dicts(dict1, dict2)
for linea in difs:
    print(linea)

In [ ]:
from autophot.prep_input import load
import shutil
auto2 = load()



Default input loaded in from: 
/home/knowogrodski/Documentos/observation_HSH/process_HSH_SN/autophot/autophot/databases/default_input.yml


{'fits_dir': None,
 'method': 'sp',
 'ignore_no_telescop': False,
 'outdir_name': 'REDUCED',
 'outcsv_name': 'REDUCED',
 'ignore_no_filter': True,
 'restart': False,
 'select_filter': False,
 'do_filter': [None],
 'target_name': None,
 'target_ra': None,
 'target_dec': None,
 'plot_source_selection': True,
 'preprocessing': {'trim_edges': False,
  'trim_edges_pixels': 50,
  'mask_sources': False,
  'mask_sources_RADEC_R': None},
 'photometry': {'do_ap_phot': False,
  'force_psf': False,
  'use_local_stars': False,
  'use_local_stars_for_FWHM': False,
  'use_local_stars_for_PSF': False,
  'use_source_arcmin': 4,
  'local_radius': 1500,
  'find_optimum_radius': False,
  'check_nyquist': True,
  'nyquist_limit': 3,
  'ap_size': 1.7,
  'inf_ap_size': 2.5,
  'ap_corr_sigma': 3,
  'ap_corr_plot': False,
  'r_in_size': 2,
  'r_out_size': 3},
 'templates': {'use_user_template': True},
 'wcs': {'allow_wcs_recheck': False,
  'remove_wcs': False,
  'force_wcs_redo': False,
  'solve_field_exe_loc'

In [ ]:
import os
import pandas as pd
from copy import deepcopy
from autophot.prep_input import load
import shutil
autophot_input = load()

# -------------------------
# Parámetros
# -------------------------
BASE = "/home/knowogrodski/Documentos/observation_HSH/process_HSH_SN"
noche = "20250823bis"
objname = "OGLE-2025-BLG-0675"
#objname = "OGLE-2025-GD-0001"

# -------------------------
# Coordenadas del target
coords_csv = os.path.join(BASE, "data", "objetos.csv")
coords = pd.read_csv(coords_csv)
row = coords[coords["objeto"] == objname]
assert not row.empty, f"{objname} no está en objetos.csv"
comb_files = [f for f in os.listdir(os.path.join(BASE, "data", noche, objname)) if "comb" in f]
assert len(comb_files)!=0, "Falta hacer combinación"
os.makedirs(os.path.join(BASE, "data", noche, objname, "phot"), exist_ok=True)
for f in comb_files:
    shutil.copy(os.path.join(BASE, "data", noche, objname,f),os.path.join(BASE, "data", noche, objname, "phot",f))

refcat_csv = os.path.join(BASE, "data", "objetos", objname, f"{objname}_cat.csv")
#refcat_csv = os.path.join(BASE, "axel0675_catBVRI.csv")

# -------------------------
# Paths absolutos
# -------------------------
fits_dir = os.path.join(BASE, "data", noche, objname, "phot")
wdir = os.path.join(BASE, "outputs", noche, objname)

os.makedirs(wdir, exist_ok=True)

# -------------------------
# Chequeos 
# -------------------------
assert os.path.isdir(fits_dir), f"No existe fits_dir: {fits_dir}"
assert os.path.isfile(coords_csv), f"No existe {coords_csv}"
assert os.path.isfile(refcat_csv), f"No existe {refcat_csv}"

fits_files = [f for f in os.listdir(fits_dir) if f.lower().endswith((".fits", ".fit"))]
assert len(fits_files) > 0, f"No hay FITS en {fits_dir}"



ra = float(row["ra_deg"].values[0])/15 # /15 to transform to hourangles
dec = float(row["dec_deg"].values[0])

# -------------------------
# Autophot input
# -------------------------

autophot_input["wdir"] = wdir
autophot_input["fits_dir"] = fits_dir

autophot_input["target_name"] = objname
autophot_input["target_ra"] = ra
autophot_input["target_dec"] = dec

autophot_input["catalog"]["use_catalog"] = "custom"
autophot_input["catalog"]["catalog_custom_fpath"] = refcat_csv
autophot_input["catalog"]["catalog_radius"] = 0.07
autophot_input['photometry']['snr_calib'] = 10
# autophot_input['catalog']['catalog'] = "apass"
autophot_input['photometry']['do_ap_phot'] = True
autophot_input['photometry']['ap_size'] = 1.5
autophot_input['plot_source_selection'] = False

#autophot_input['select_filter'] = True
#autophot_input['do_filter'] = ["I"]
# autophot_input['photometry']['find_optimum_radius'] =True

# autophot_input['psf']['isolate_sources'] = False          # Desactiva el chequeo de fuentes cercanas débiles
# autophot_input['psf']['min_snr_psf'] = 50                 # Baja el SNR mínimo para estrellas PSF (default suele ser alto)
# autophot_input['psf']['max_snr_psf'] = 1000                # Evita saturadas
# autophot_input['catalog']['faint_mag_cut'] = 18            # Limita a estrellas más brillantes para PSF
# === Mejoras en calibración (reduce scatter alto) ===

autophot_input['catalog']['catalog_radius'] = 0.1  # Aumenta ligeramente para más estrellas, pero no >0.15 (evita más crowding)
autophot_input['catalog']['matching_source_FWHM_limit'] = 40  # Baja para ser más tolerante con FWHM variables
autophot_input['catalog']['catalog_matching_limit'] = 17  # Menos matches para calidad
autophot_input['catalog']['max_catalog_sources'] = 100  # Limita a ~100 para evitar saturadas/blendadas

# === Fotometría: Fuerza PSF (mejor para crowding que aperture) ===
autophot_input['photometry']['do_ap_phot'] = False  # Desactiva aperture como principal
autophot_input['photometry']['force_psf'] = False # Fuerza PSF
autophot_input['photometry']['snr_calib'] = 5       # Baja SNR corte para calibración (usa más estrellas, pero chequea manual)
# autophot_input['photometry']['find_optimum_radius'] = True
autophot_input['photometry']['ap_size'] = 1.1
# === PSF settings (evita fallos por "faint sources near") ===
autophot_input['psf']['psf_source_no'] = 30          # Usa más estrellas para modelo PSF
autophot_input['psf']['min_psf_source_no'] = 5       # Mínimo para construir
autophot_input['psf']['construction_SNR'] = 6       # Baja para incluir más
autophot_input['psf']['plot_PSF_residuals'] = True   # Activa plots para inspeccionar residuals (busca blending)

# === Source detection: Más estricto para outliers ===
autophot_input['source_detection']['threshold_value'] = 10  # Baja de 25 para detectar más, pero filtra después
autophot_input['source_detection']['remove_sat'] = True
autophot_input['source_detection']['pix_bound'] = 50  # Más margen de bordes

# === Zeropoint: Usa media y sigma clip más agresivo ===
autophot_input['zeropoint']['zp_sigma'] = 2.0  # Sigma clip más estricto (quita outliers)
autophot_input['zeropoint']['zp_use_mean'] = True
autophot_input['zeropoint']['matching_source_SNR_limit'] = 10  # Baja para más puntos en ZP

# === Otros útiles ===
autophot_input['limiting_magnitude']['skip_lmag'] = False  # Calcula limiting mag para chequear
autophot_input['wcs']['search_radius'] = 0.3  # Aumenta si WCS falla
autophot_input['cosmic_rays']['remove_cmrays'] = True

# === Clave: ser menos estricto con el aislamiento de estrellas PSF ===
autophot_input['psf']['construction_SNR'] = 3          # Baja de 10 → permite estrellas más débiles
autophot_input['psf']['psf_source_no'] = 20           # Intenta usar hasta 20 (default 15)
autophot_input['psf']['min_psf_source_no'] = 3        # Mínimo aceptable

# === Relaja el rechazo por fuentes débiles cercanas (principal causa del fallo) ===
# No hay parámetro directo, pero bajar el umbral de detección ayuda indirectamente
autophot_input['source_detection']['threshold_value'] = 8   # De 10 → detecta más, pero el aislamiento es menos estricto
autophot_input['source_detection']['isolate_sources_fwhm_sep'] = 2  # De 5 → permite estrellas más cerca (menos rechazos)

# === Zeropoint más robusto ===
autophot_input['zeropoint']['matching_source_SNR_limit'] = 8   # De 10 → usa más estrellas en calibración
autophot_input['zeropoint']['zp_sigma'] = 3.0                  # Clip menos agresivo

# === Catálogo: un poco más grande para tener más candidatas PSF ===
autophot_input['catalog']['catalog_radius'] = 0.15   # De 0.1 → más estrellas brillantes posibles
autophot_input['catalog']['max_catalog_sources'] = 150
# Mueve el annulus más lejos para 
# +++++++evitar esa estrella contaminante
autophot_input['photometry']['r_in_size'] = 3    # Era 2 → radio interno más grande
autophot_input['photometry']['r_out_size'] = 5   # Era 3 → anillo más externo y ancho

# Opcional: usa background local más robusto (recomendado en campos crowded)
autophot_input['fitting']['remove_bkg_local'] = True
autophot_input['fitting']['remove_bkg_surface'] = False
autophot_input['fitting']['bkg_level'] = 3         # Baja un poco si usas local
autophot_input['target_photometry']['adjust_SN_loc'] = False,

# -------------------------
# Log rápido
# -------------------------
print(f"WDIR     : {wdir}")
print(f"FITS DIR : {fits_dir}")
print(f"CATALOG  : {refcat_csv}")
print(f"TARGET   : {objname}")
print(f"N FITS   : {len(fits_files)}")



Default input loaded in from: 
/home/knowogrodski/Documentos/observation_HSH/process_HSH_SN/autophot/autophot/databases/default_input.yml


FileNotFoundError: [Errno 2] No such file or directory: '/home/knowogrodski/Documentos/observation_HSH/process_HSH_SN/data/objetos.csv'

In [6]:
from autophot.prep_input import load
import shutil
autophot_input = load()
autophot_input
for k in autophot_input.keys():
    if isinstance(autophot_input[k], dict):
        print(f"  {k}:")
        for kk in autophot_input[k].keys():
            print(f"    {kk}: {autophot_input[k][kk]}")
 
    else:
        print(f"  {k}: {autophot_input[k]}")
        
        

Default input loaded in from: 
/home/knowogrodski/Documentos/observation_HSH/process_HSH_SN/autophot/autophot/databases/default_input.yml
  fits_dir: None
  method: sp
  ignore_no_telescop: False
  outdir_name: REDUCED
  outcsv_name: REDUCED
  ignore_no_filter: True
  restart: False
  select_filter: False
  do_filter: [None]
  target_name: None
  target_ra: None
  target_dec: None
  plot_source_selection: True
  preprocessing:
    trim_edges: False
    trim_edges_pixels: 50
    mask_sources: False
    mask_sources_RADEC_R: None
  photometry:
    do_ap_phot: False
    force_psf: False
    use_local_stars: False
    use_local_stars_for_FWHM: False
    use_local_stars_for_PSF: False
    use_source_arcmin: 4
    local_radius: 1500
    find_optimum_radius: False
    check_nyquist: True
    nyquist_limit: 3
    ap_size: 1.7
    inf_ap_size: 2.5
    ap_corr_sigma: 3
    ap_corr_plot: False
    r_in_size: 2
    r_out_size: 3
  templates:
    use_user_template: True
  wcs:
    allow_wcs_recheck

In [29]:
autophot_input["outcsv_name"], autophot_input["outdir_name"]

('REDUCED', 'REDUCED')

In [ ]:

BASE = Path(cfg["paths"]["BASE"])
data_path = Path(BASE, cfg["paths"]["data_dir"])
object_path = Path(data_path, cfg["paths"]["objects_dir"])
night = cfg["night"]
night_dir = Path(data_path, night)
objects_csv = Path(object_path, "objetos.csv")
images_file_name = cfg["path"]["images_data_file"]
images_file = Path(night_dir, images_file_name)
fits_dir = os.path.join(BASE, "data", noche, objname, f"phot_{noche}")
wdir = os.path.join(BASE, "outputs", noche, objname)




{'method': 'sp',
 'ignore_no_telescop': False,
 'outdir_name': 'lightcurve',
 'outcsv_name': 'REDUCED',
 'ignore_no_filter': True,
 'restart': False,
 'select_filter': False,
 'do_filter': ['None'],
 'plot_source_selection': True,
 'preprocessing': {'trim_edges': False,
  'trim_edges_pixels': 50,
  'mask_sources': False,
  'mask_sources_RADEC_R': 'None'},
 'photometry': {'do_ap_phot': False,
  'force_psf': False,
  'use_local_stars': False,
  'use_local_stars_for_FWHM': False,
  'use_local_stars_for_PSF': False,
  'use_source_arcmin': 4,
  'local_radius': 1500,
  'find_optimum_radius': False,
  'check_nyquist': True,
  'nyquist_limit': 3,
  'ap_size': 1.7,
  'inf_ap_size': 2.5,
  'ap_corr_sigma': 3,
  'ap_corr_plot': False,
  'r_in_size': 2,
  'r_out_size': 3},
 'templates': {'use_user_template': True},
 'wcs': {'allow_wcs_recheck': False,
  'remove_wcs': False,
  'force_wcs_redo': False,
  'solve_field_exe_loc': 'None',
  'offset_param': 5.0,
  'search_radius': 0.25,
  'downsample': 2